In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.signal import find_peaks
import pandas as pd

In [ ]:
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "deeplab-insect-locomotion-analysis":
    for parent in BASE_DIR.parents:
        if parent.name == "deeplab-insect-locomotion-analysis":
            BASE_DIR = parent
            print(f"Base directory found: {parent}")
            break

# Setting the current working directory to the base directory of the project (pwd so even Path.cwd() will return the base directory)
os.chdir(BASE_DIR)

from helper import parse_file_trial_duration_info, filter_dataframe

In [ ]:
# Reset to default white theme and explicitly set face colors instead of using system theme colors
plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

In [ ]:
trial_burst_map = {
    "0.0": {"trial_detail": "No Stimulation", "color": "#a734d5"},
    "0.1": {"trial_detail": "0.1s with 1.0s period (T)", "color": "#b3cde0"},
    "0.2": {"trial_detail": "0.2s with 1.0s period (T)", "color": "#6497b1"},
    "0.3": {"trial_detail": "0.3s with 1.0s period (T)", "color": "#005b96"},
    "0.4": {"trial_detail": "0.4s with 1.0s period (T)", "color": "#03396c"},
    "0.5": {"trial_detail": "0.5s with 1.0s period (T)", "color": "#011f4b"},
}

In [ ]:
EX_16_FILES_DIR = [str(path) for path in Path("videos_jetson_experiment_16").glob("*.csv")]  # Experiment 16 (Retractor), Position 1, Female


In [ ]:
def processing_data_for_plotting(files_list):
    parsed_files = []
    for file in files_list:
        info = parse_file_trial_duration_info(file)
        info["filename"] = file
        parsed_files.append(info)

    sorted_files = sorted(parsed_files, key=lambda file: (file["ind_num"], file["formatted_dur_num"]))
    unique_burst = sorted(set(file["formatted_dur_num"] for file in sorted_files))
    num_burst = len(unique_burst)
    processed_data = []
    ind_num = sorted_files[0]["ind_num"]  # Initialize ind_num outside the loop

    for info in sorted_files:
        df = pd.read_csv(info["filename"])
        df_filtered = filter_dataframe(df, include_likelihood=True)

        # Pre-processing data for stimulation ON
        condition = df_filtered["PWM_Light_likelihood"].between(0.4, 1.0)
        df_filtered["PWM_Time_Period"] = (condition != condition.shift()).cumsum()

        spans = (
            df_filtered[condition]
            .groupby("PWM_Time_Period")
            .agg(start_frame=("frame", "min"), end_frame=("frame", "max"), start_time=("time", "min"), end_time=("time", "max"))
            .reset_index(drop=True)
        )

        angle_signal = df_filtered["angle_from_initial"].dropna().values
        peaks, peak_properties = find_peaks(angle_signal, prominence=1)
        peak_heights = angle_signal[peaks]

        processed_data.append(
            {
                "trial_num": info["trial_num"],
                "df_filtered": df_filtered,
                "peaks": peaks,
                "peak_properties": peak_properties,
                "peak_heights": peak_heights,
                "burst_dur": info["formatted_dur_num"],
                "spans": spans,
            }
        )

    return processed_data, unique_burst, num_burst, ind_num

In [ ]:
# Plotting angle over time for each individual and plots all the trials in a ascending order of the duration time.
def plot_angle_over_time(files):
    """Plot angle over time for each individual and plots all the trials in a ascending order of the duration time.
    Args:
        files (list): List of file paths to the CSV files for each trial.

    Returns:
        None: Displays the plots and saves them as PNG files.
    """
    processed_data, unique_burst, num_burst, ind_num = processing_data_for_plotting(files)

    # Plot angle over time
    fig, axs = plt.subplots(num_burst, 1, figsize=(16, 4 * num_burst), layout="constrained")

    if num_burst == 1:
        axs = [axs]

    print(f"Unique bursts for individual {ind_num}: {unique_burst}")

    burst_to_ax_map = {burst: ax for burst, ax in zip(unique_burst, axs)}

    for trial_data in processed_data:
        trial_num = trial_data["trial_num"]
        df_filtered = trial_data["df_filtered"]
        peaks = trial_data["peaks"]
        burst_dur = trial_data["burst_dur"]
        spans = trial_data["spans"]

        # Get the specific subplot axis assigned to this burst duration
        ax = burst_to_ax_map[burst_dur]

        # filter the first times without stimulation
        if spans.empty:
            print(f"Warning: No stimulation spans found in Trial {trial_num}. Using fallback start time.")
            start_time = df_filtered["time"].iloc[0]
        else:
            start_time = spans["start_time"].iloc[0]

        filtered_timeseries_data = df_filtered[df_filtered["time"] >= start_time]
        shifted_time = filtered_timeseries_data["time"] - start_time

        # Color palette for the trials based on burst duration
        mapped_color = trial_burst_map[str(burst_dur)]["color"]

        # Plot the current trial onto the designated burst duration subplot (overlapping automatically)
        ax.plot(
            shifted_time,
            filtered_timeseries_data["angle_from_initial"],
            label=f"Trial {trial_num}, Burst: {burst_dur}s",
            color=mapped_color,
            alpha=0.7,
        )
        ax.plot(df_filtered["time"].iloc[peaks] - start_time, df_filtered["angle_from_initial"].iloc[peaks], "ro", markersize=5)

        ax.vlines(
            x=df_filtered["time"].iloc[peaks] - start_time,
            ymin=0,
            ymax=df_filtered["angle_from_initial"].iloc[peaks],
            color="red",
            alpha=0.3,
        )

        # Highlight the time series when electrical stimulation is active (PMW Light ON)
        for i, row in spans.iterrows():
            ax.axvspan(row["start_time"] - start_time, row["end_time"] - start_time, color="blue", alpha=0.03)

        ax.legend(loc="upper right", fontsize=8)

        if not ax.get_title():
            ax.set_title(f"Burst Duration: {burst_dur}, Frequency: 100Hz, Duty Ratio: 50%", fontsize=12)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Angle (degrees)")
            ax.grid(True, alpha=0.3)

        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc="upper right", fontsize=8)

    fig.suptitle(f"Individual {ind_num} - Angle Changes Across All Durations", fontsize=16, fontweight="bold")
    plt.savefig(f"ind{ind_num}_all_trials.png", dpi=300)
    plt.show()


In [ ]:
def plot_peak_height(files_list):
    processed_data, unique_burst, num_burst, ind_num = processing_data_for_plotting(files_list)

    # Processing data for the boxpllot, peak height quantification
    plot_data = []
    for trial_data in processed_data:
        trial_num = trial_data["trial_num"]
        df_filtered = trial_data["df_filtered"]
        peaks = trial_data["peaks"]
        burst_dur = trial_data["burst_dur"]
        peak_angles = df_filtered["angle_from_initial"].iloc[peaks]

        if len(peak_angles) > 0:
            for angle in peak_angles:
                plot_data.append(
                    {
                        "Trial": f" {trial_num}",
                        "Peak Angle": angle,
                        "color": trial_burst_map[str(burst_dur)]["color"],
                        "trial_legend": trial_burst_map[str(burst_dur)]["trial_detail"],
                    }
                )

    df = pd.DataFrame(plot_data)
    color_palette = dict(zip(df["trial_legend"], df["color"]))

    plt.figure(figsize=(18, 4))
    sns.boxplot(data=df, x="Trial", y="Peak Angle", width=0.7, hue="trial_legend", palette=color_palette, boxprops=dict(alpha=0.5))
    plt.xticks(rotation=45, ha="right")
    # Position and format the legend
    plt.legend(title="Trial Detail", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

    plt.savefig(f"ind{ind_num}_all_trials_for_each_duration_initial_angle.png", dpi=300)
    plt.show()

In [ ]:
def plot_one_period(files_list):
    """Plot only one period of the angle changes for all the trials in a single plot, to compare the angle changes across different burst duration.
    Args:
        files_list (list): List of file paths to the CSV files for each trial.
    Returns:
        None: Displays the plot and saves it as a PNG file.
    """
    processed_data, unique_burst, num_burst, ind_num = processing_data_for_plotting(files_list)

    fig, ax = plt.subplots()
    for trial_data in processed_data:
        df_filtered = trial_data["df_filtered"]
        spans = trial_data["spans"]
        burst_dur = trial_data["burst_dur"]
        trial_num = trial_data["trial_num"]

        if len(spans) < 2:
            print(f"Skipping Trial {trial_num}: Needs at least 2 stimulation periods to plot a span.")
            continue

        first_period = spans["start_time"].iloc[1]
        start_time = spans["start_time"].iloc[0]

        filtered_timeseries_data = df_filtered[(df_filtered["time"] >= start_time) & (df_filtered["time"] <= first_period)]
        shifted_time = filtered_timeseries_data["time"] - start_time

        mapped_color = trial_burst_map[str(burst_dur)]["color"]

        # Plot the current trial onto the designated burst subplot (overlapping automatically)
        ax.plot(shifted_time, filtered_timeseries_data["angle_from_initial"], label=f"{burst_dur} s", color=mapped_color, alpha=0.7)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc="upper right", fontsize=8)

    ax.set_ylabel("Angle (degrees)")
    ax.set_xlabel("Time (s)")
    ax.set_ylim(0.5, 1.5)
    ax.set_title("Joint Displacement for First Stimulation Period")

    plt.savefig(f"ind{ind_num}_burst_all.png", dpi=300)
    plt.show()

In [ ]:
plot_angle_over_time(EX_16_FILES_DIR)

In [ ]:
plot_peak_height(EX_16_FILES_DIR)

In [ ]:
plot_one_period(EX_16_FILES_DIR)